# LangChain Baseline Translation Pipeline Demo

Run the single-prompt `LangChainBasePipeline` end to end for a quick ablation-style translation baseline.

## 1️⃣ Setup Environment

In [16]:
# Core imports
from pathlib import Path
import json
import importlib
from datetime import datetime, timezone
from time import perf_counter
from typing import Any

try:
    import ollama
except ImportError as exc:  # pragma: no cover - notebook runtime guard
    raise RuntimeError(
        "The `ollama` package is required to resolve local models. "
        "Install it with `pip install ollama` and ensure the Ollama service is running."
    ) from exc

try:
    from langchain_ollama import ChatOllama
except ImportError as exc:  # pragma: no cover - notebook runtime guard
    raise RuntimeError(
        "The `langchain-ollama` package is required. Install it with `pip install langchain-ollama`."
    ) from exc

from wordnet_autotranslate.utils.log_utils import sanitize_model_name
import wordnet_autotranslate.pipelines.langchain_base_pipeline as base_module

# Reload to pick up recent edits if the notebook is re-run after development changes
base_module = importlib.reload(base_module)
LangChainBasePipeline = base_module.LangChainBasePipeline

print("✅ Imports complete")

✅ Imports complete


## 2️⃣ Load Synset Dataset

In [17]:
DATA_PATH = Path("../examples/serbian_english_synset_pairs_enhanced.json")
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Expected dataset at {DATA_PATH.resolve()} — adjust DATA_PATH or regenerate the file."
    )

with DATA_PATH.open("r", encoding="utf-8") as f:
    dataset = json.load(f)

pairs_raw = dataset.get("pairs")
if not isinstance(pairs_raw, list):
    raise ValueError("Dataset missing a 'pairs' list; verify the JSON structure.")

synset_pairs: list[dict[str, Any]] = pairs_raw
ALL_SYNSETS: list[dict[str, Any]] = []
DISPLAY_NAMES: list[str] = []

for pair in synset_pairs:
    lemmas = pair.get("english_lemmas") or []
    if isinstance(lemmas, str):
        lemmas = [lemmas]
    examples = pair.get("english_examples") or []
    if isinstance(examples, str):
        examples = [examples]
    synset = {
        "id": pair.get("english_id"),
        "lemmas": list(lemmas),
        "definition": pair.get("english_definition"),
        "examples": list(examples),
        "pos": pair.get("english_pos"),
    }
    ALL_SYNSETS.append(synset)

    display_name = pair.get("english_name") or ", ".join(synset["lemmas"][:2])
    if not display_name:
        display_name = synset.get("id") or "(unknown synset)"
    DISPLAY_NAMES.append(display_name)

print(f"✅ Loaded {len(ALL_SYNSETS)} synset pairs from {DATA_PATH}")
if ALL_SYNSETS:
    preview_total = min(3, len(ALL_SYNSETS))
    print(f"   Previewing the first {preview_total} entries:")
    for idx, synset in enumerate(ALL_SYNSETS[:preview_total], start=1):
        printable_definition = (synset.get("definition") or "")[:80]
        if synset.get("definition") and len(synset["definition"]) > 80:
            printable_definition += "…"
        print(f"   {idx}. {synset.get('id')} — {', '.join(synset.get('lemmas', [])[:3])}")
        print(f"      Def: {printable_definition}")

✅ Loaded 27 synset pairs from ..\examples\serbian_english_synset_pairs_enhanced.json
   Previewing the first 3 entries:
   1. ENG30-03574555-n — institution
      Def: an establishment consisting of a building or complex of buildings where an organ…
   2. ENG30-07810907-n — condiment
      Def: a preparation (a sauce or relish or spice) to enhance flavor or enjoyment
   3. ENG30-01376245-v — scatter, sprinkle, dot
      Def: distribute loosely


## 3️⃣ Resolve Ollama Model

In [18]:
PREFERRED_MODEL = "gpt-oss:120b"
TEMPERATURE = 0.0
TIMEOUT = 120

try:
    model_listing = ollama.list()
except Exception as exc:  # pragma: no cover - notebook runtime guard
    raise RuntimeError(
        "Unable to communicate with the Ollama service. Ensure it is running locally."
    ) from exc

available_models = sorted(
    {
        getattr(model_entry, "model", None)
        for model_entry in getattr(model_listing, "models", [])
        if getattr(model_entry, "model", None)
    }
)

if not available_models:
    raise RuntimeError(
        "No Ollama models available. Pull at least one model (e.g., `ollama pull gemma3:27b`)."
    )

lookup = {name.lower(): name for name in available_models}
requested_model = (PREFERRED_MODEL or "").strip()
resolution_strategy = "preferred"
fallback_reason = None

if requested_model and requested_model.lower() in lookup:
    resolved_model = lookup[requested_model.lower()]
else:
    resolved_model = available_models[0]
    resolution_strategy = "fallback_first_available"
    if requested_model:
        fallback_reason = (
            f"Preferred model '{requested_model}' not found; using '{resolved_model}'."
        )
    else:
        fallback_reason = "No preferred model specified; using first available model."

MODEL_RESOLUTION = {
    "requested": requested_model or resolved_model,
    "resolved": resolved_model,
    "resolved_safe": sanitize_model_name(resolved_model),
    "fallback_used": resolution_strategy != "preferred",
    "resolution_strategy": resolution_strategy,
    "available_models": available_models,
}
if fallback_reason:
    MODEL_RESOLUTION["reason"] = fallback_reason

if MODEL_RESOLUTION["fallback_used"] and fallback_reason:
    print(f"⚠️ {fallback_reason}")

print(
    "✅ Using model:",
    MODEL_RESOLUTION["resolved"],
    "| Available:",
    ", ".join(available_models),
)


✅ Using model: gpt-oss:120b | Available: deepseek-r1:70b, deepseek-r1:8b, deepseek-r1:latest, deepseek-v3.1:671b, gemma3:1b, gemma3:27b, gpt-oss:120b, gpt-oss:20b, qwen3:235b, qwen3:30b


## 4️⃣ Initialize Base Pipeline

In [19]:
llm = ChatOllama(
    model=MODEL_RESOLUTION["resolved"],
    temperature=TEMPERATURE,
    timeout=TIMEOUT,
)

pipeline = LangChainBasePipeline(
    source_lang="en",
    target_lang="sr",
    llm=llm,
    model=MODEL_RESOLUTION["resolved"],
    model_metadata=dict(MODEL_RESOLUTION),
)

print("✅ LangChainBasePipeline ready for translation")

✅ LangChainBasePipeline ready for translation


## 5️⃣ Run Single Synset Translation

In [20]:
if not ALL_SYNSETS:
    raise RuntimeError("Dataset not loaded. Run the dataset cell first.")

SINGLE_INDEX = 0
if SINGLE_INDEX >= len(ALL_SYNSETS):
    raise IndexError(
        f"SINGLE_INDEX {SINGLE_INDEX} is out of range for {len(ALL_SYNSETS)} synsets."
    )

synset_payload = ALL_SYNSETS[SINGLE_INDEX]
SINGLE_PAIR = synset_pairs[SINGLE_INDEX]

lemmas_display = ", ".join(synset_payload.get("lemmas", [])) or "(no lemmas)"
print(f"🔄 Translating synset {synset_payload.get('id')} ({lemmas_display})")
run_start = perf_counter()
SINGLE_RESULT = pipeline.translate_synset(synset_payload)
SINGLE_EXECUTION_SECONDS = perf_counter() - run_start

print(f"✅ Translation complete in {SINGLE_EXECUTION_SECONDS:.2f} seconds")

🔄 Translating synset ENG30-03574555-n (institution)
✅ Translation complete in 1521.86 seconds
✅ Translation complete in 1521.86 seconds


## 6️⃣ Translate Entire Dataset

Run the baseline pipeline across every synset in the dataset. If you've already executed the single-synset cell above, its result is reused to save one model call.

In [21]:
if not ALL_SYNSETS:
    raise RuntimeError("Dataset not loaded. Run the dataset cell first.")

BATCH_RECORDS: list[dict[str, Any]] = []
BATCH_TOTAL_SECONDS: float
synset_execution_seconds: dict[str, float] = {}
single_index = globals().get("SINGLE_INDEX", 0)

print(f"🔄 Translating all {len(ALL_SYNSETS)} synsets with LangChainBasePipeline\n")
batch_start = perf_counter()
for idx, (synset, display_name) in enumerate(zip(ALL_SYNSETS, DISPLAY_NAMES), start=1):
    synset_id = synset.get("id") or f"synset_{idx}"
    reuse_single = False
    if (
        'SINGLE_RESULT' in globals()
        and 'SINGLE_EXECUTION_SECONDS' in globals()
        and idx - 1 == single_index
    ):
        reuse_single = True

    if reuse_single:
        result = SINGLE_RESULT
        duration = SINGLE_EXECUTION_SECONDS
    else:
        synset_timer = perf_counter()
        result = pipeline.translate_synset(synset)
        duration = perf_counter() - synset_timer

    synset_execution_seconds[synset_id] = duration
    BATCH_RECORDS.append(
        {
            "synset": synset,
            "display_name": display_name,
            "result": result,
            "seconds": duration,
            "reused_single": reuse_single,
        }
    )

    retained = len(result.get("translated_synonyms") or [])
    status_marker = "⏪ reused single-run" if reuse_single else "⏱️ new call"
    print(f"[{idx}/{len(ALL_SYNSETS)}] {synset_id} — {display_name}")
    print(f"   {status_marker}; runtime {duration:.2f}s; kept {retained} synonyms\n")

BATCH_TOTAL_SECONDS = perf_counter() - batch_start
print(f"✅ Completed {len(BATCH_RECORDS)} translations in {BATCH_TOTAL_SECONDS:.2f} seconds")

🔄 Translating all 27 synsets with LangChainBasePipeline

[1/27] ENG30-03574555-n — institution.n.02
   ⏪ reused single-run; runtime 1521.86s; kept 2 synonyms

[2/27] ENG30-07810907-n — condiment.n.01
   ⏱️ new call; runtime 28.60s; kept 3 synonyms

[2/27] ENG30-07810907-n — condiment.n.01
   ⏱️ new call; runtime 28.60s; kept 3 synonyms

[3/27] ENG30-01376245-v — scatter.v.03
   ⏱️ new call; runtime 57.98s; kept 5 synonyms

[3/27] ENG30-01376245-v — scatter.v.03
   ⏱️ new call; runtime 57.98s; kept 5 synonyms

[4/27] ENG30-01382083-v — pick.v.02
   ⏱️ new call; runtime 30.23s; kept 4 synonyms

[4/27] ENG30-01382083-v — pick.v.02
   ⏱️ new call; runtime 30.23s; kept 4 synonyms

[5/27] ENG30-01393996-v — sweep.v.06
   ⏱️ new call; runtime 35.75s; kept 3 synonyms

[5/27] ENG30-01393996-v — sweep.v.06
   ⏱️ new call; runtime 35.75s; kept 3 synonyms

[6/27] ENG30-01421622-v — insert.v.01
   ⏱️ new call; runtime 57.03s; kept 4 synonyms

[6/27] ENG30-01421622-v — insert.v.01
   ⏱️ new call; ru

## 7️⃣ Inspect Translation Output

Use the helper below to review a specific synset. Update `SINGLE_INDEX` (0-based) before re-running to browse other entries.

In [22]:
def display_result(result: dict[str, Any]) -> None:
    translation = result.get("translation") or "—"
    definition = result.get("definition_translation") or "—"
    synonyms = result.get("translated_synonyms") or []
    summary = result.get("curator_summary") or "(no curator summary)"

    print("Representative literal:", translation)
    print("Definition (sr):", definition)
    if synonyms:
        print("Synonyms:")
        for idx, word in enumerate(synonyms, start=1):
            print(f"  {idx}. {word}")
    else:
        print("Synonyms: —")

    print("\nCurator summary:")
    print(summary)


def show_synset(index: int = 0) -> None:
    if 'BATCH_RECORDS' in globals() and BATCH_RECORDS:
        if index >= len(BATCH_RECORDS):
            raise IndexError(f"Index {index} out of range for {len(BATCH_RECORDS)} translations.")
        record = BATCH_RECORDS[index]
        synset = record["synset"]
        print(f"📌 Synset {index + 1}/{len(BATCH_RECORDS)} — {synset.get('id')} ({record['display_name']})")
        print(f"   Runtime: {record['seconds']:.2f}s; reused single result: {record['reused_single']}")
        display_result(record["result"])
    elif 'SINGLE_RESULT' in globals():
        print("📌 Showing single-synset result only (batch not executed yet).")
        display_result(SINGLE_RESULT)
    else:
        raise RuntimeError("No translation results available. Run the translation cells first.")


show_synset(SINGLE_INDEX)

📌 Synset 1/27 — ENG30-03574555-n (institution.n.02)
   Runtime: 1521.86s; reused single result: True
Representative literal: institucija
Definition (sr): ustanova koja se sastoji od jedne zgrade ili kompleksa zgrada u kojima se nalazi organizacija za promovisanje nekog cilja
Synonyms:
  1. institucija
  2. ustanova

Curator summary:
Primary literal: institucija
Synonyms: institucija, ustanova
Gloss: ustanova koja se sastoji od jedne zgrade ili kompleksa zgrada u kojima se nalazi organizacija za promovisanje nekog cilj…


## 8️⃣ Persist Minimal Logs

Save a compact JSON bundle containing metadata, timings, and sanitized payloads for every translated synset.

In [23]:
OUTPUT_DIR = Path("output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if 'BATCH_RECORDS' not in globals() or not BATCH_RECORDS:
    raise RuntimeError("No batch translation results found. Run the batch translation cell first.")

model_safe = MODEL_RESOLUTION.get("resolved_safe") or sanitize_model_name(MODEL_RESOLUTION["resolved"])
timestamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
export_path = OUTPUT_DIR / f"baseline_pipeline_{model_safe}_{timestamp}.json"


def sanitize_for_json(value: Any) -> Any:
    """Best-effort conversion of complex objects into JSON-serializable data."""
    if value is None or isinstance(value, (str, int, float, bool)):
        return value
    if isinstance(value, dict):
        return {str(key): sanitize_for_json(subvalue) for key, subvalue in value.items()}
    if isinstance(value, (list, tuple, set)):
        return [sanitize_for_json(item) for item in value]

    message_type = getattr(value, "type", None)
    content = getattr(value, "content", None)
    if message_type is not None and content is not None:
        return {
            "type": str(message_type),
            "content": sanitize_for_json(content),
        }

    if hasattr(value, "__dict__"):
        return sanitize_for_json(vars(value))

    return str(value)


entries: list[dict[str, Any]] = []
for record in BATCH_RECORDS:
    synset = record["synset"]
    result = record["result"]
    payload = sanitize_for_json(result.get("payload"))

    entries.append(
        {
            "synset": {
                "id": synset.get("id"),
                "lemmas": list(synset.get("lemmas", [])),
                "definition": synset.get("definition"),
                "examples": list(synset.get("examples", [])),
                "pos": synset.get("pos"),
                "display_name": record.get("display_name"),
            },
            "result": {
                "translation": result.get("translation"),
                "definition_translation": result.get("definition_translation"),
                "translated_synonyms": list(result.get("translated_synonyms") or []),
                "curator_summary": result.get("curator_summary"),
                "model": result.get("model"),
                "notes": result.get("notes"),
                "examples": list(result.get("examples") or []),
                "payload": payload,
                "raw_response": result.get("raw_response"),
            },
            "timing": {
                "seconds": record.get("seconds"),
                "reused_single": record.get("reused_single"),
            },
        }
    )

metadata = {
    "model": MODEL_RESOLUTION["resolved"],
    "model_resolution": dict(MODEL_RESOLUTION),
    "timestamp_utc": timestamp,
    "source_lang": pipeline.source_lang,
    "target_lang": pipeline.target_lang,
    "total_synsets": len(entries),
    "batch_total_seconds": BATCH_TOTAL_SECONDS,
}

export_payload = {
    "metadata": metadata,
    "entries": entries,
}

with export_path.open("w", encoding="utf-8") as f:
    json.dump(export_payload, f, ensure_ascii=False, indent=2)

print(f"💾 Saved baseline log to {export_path.resolve()}")
print("   Entries captured:", len(entries))

💾 Saved baseline log to E:\Github\wordnet_autotranslate\notebooks\output\baseline_pipeline_gpt-oss-120b_20251025T194126Z.json
   Entries captured: 27
